# Python for AI — Class 8
### Recursion, decorators and generators

Class 7 treated functions as values you can pass around — that's what `lambda` gave you. Today functions do more: they call *themselves* (recursion), *wrap* other functions (decorators), and hand out values one at a time (generators).

Each one comes with real business examples — a P&L, refund permissions, an audit log, invoice numbers — and the day ends with all of it working together in one small shop program.

**How to use this notebook:** run each cell with the play button, or `Shift + Enter`.

Cells marked **BREAKS ON PURPOSE** are *supposed* to show a red error. Cells marked **WRONG ON PURPOSE** run fine and give the *wrong answer* — those are the dangerous ones. Don't fix either before class; that is the lesson.

# 1. Recursion — a function that calls itself

A recursive function does its job by handing a **smaller version of the same job** to itself. ("Job" here just means whatever you've asked the function to do — count down from 3, add up 1 to 10, reverse a word.)

Every recursive function needs two parts:

1. A **base case** — the smallest version of the job, simple enough to answer directly, with no further recursive call.
2. A **recursive case** — the function calling itself with a smaller piece of the job, moving toward the base case.

Miss the base case, and it's Day 3's infinite loop again — except this time it's an infinite chain of function calls.

### Start simple: a countdown

In [26]:
def countdown(n):
    if n == 0:                 # base case - stop here
        print("Done!")
        return
    print(n)
    countdown(n - 1)           # recursive case - the same job, one smaller

countdown(3)

3
2
1
Done!


Follow it call by call:

```
countdown(3)  prints 3, then calls countdown(2)
countdown(2)  prints 2, then calls countdown(1)
countdown(1)  prints 1, then calls countdown(0)
countdown(0)  base case: prints "Done!" and stops
```

Each call does one small piece of the work — print one number — then hands the rest of the job to a smaller copy of itself.

### Adding up 1 to n

In [ ]:
def sum_to(n):
    if n == 0:                  # base case - the sum of nothing is 0
        return 0
    return n + sum_to(n - 1)    # n, plus the sum of everything below it

print(sum_to(3))
print(sum_to(10))

```
sum_to(3) = 3 + sum_to(2)
          = 3 + 2 + sum_to(1)
          = 3 + 2 + 1 + sum_to(0)
          = 3 + 2 + 1 + 0
          = 6
```

### Reversing a word

In [27]:
def reverse(text):
    if text == "":                          # base case - nothing left to reverse
        return ""
    return reverse(text[1:]) + text[0]      # reverse the rest, then put the first letter last



print(reverse("hello"))

olleh


Trace `reverse("cat")`. Each call sets its **first** letter aside and asks a smaller copy of itself to reverse the rest:

```
reverse("cat") = reverse("at") + "c"
               = (reverse("t") + "a") + "c"
               = ((reverse("") + "t") + "a") + "c"
               = (("" + "t") + "a") + "c"          <- base case: reverse("") returns ""
               = ("t" + "a") + "c"                 <- reverse("t") returns "t"
               = "ta" + "c"                        <- reverse("at") returns "ta"
               = "tac"                             <- reverse("cat") returns "tac"
```

Nothing gets joined on the way down — every call is paused, waiting. Only once the base case hands back `""` do the answers travel back up, and each call sticks the letter it set aside onto the **end**. That's why the first letter, `c`, ends up last.

### The classic: factorial

`5!` ("5 factorial") means `5 × 4 × 3 × 2 × 1`. It has exactly the same shape as `sum_to` — multiply instead of add, and the base case returns `1` instead of `0`.

In [27]:
def factorial(n):
    if n == 0:                     # base case - stops the recursion
        return 1
    return n * factorial(n - 1)    # recursive case - a smaller job

print(factorial(5))

120


Trace `factorial(3)` by hand — it has to go all the way down to the base case before anything can be multiplied:

```
factorial(3)
= 3 * factorial(2)
    = 3 * (2 * factorial(1))
        = 3 * (2 * (1 * factorial(0)))
            = 3 * (2 * (1 * 1))         <- base case reached, returns 1
        = 3 * (2 * 1)                   <- factorial(1) returns 1
    = 3 * 2                             <- factorial(2) returns 2
= 6                                     <- factorial(3) returns 6
```

### The call stack

Each call to `factorial` pauses and waits for the call it made to finish, before it can compute its own answer. Python keeps track of every paused, waiting call in a structure called the **call stack** — literally a stack, like Day 4's `append`/`pop`: the most recent call is the first one to finish and get popped off.

Forget the base case, and that stack never stops growing:

In [28]:
# BREAKS ON PURPOSE - no base case, so it never stops calling itself
def broken_factorial(n):
    return n * broken_factorial(n - 1)

broken_factorial(5)

RecursionError: maximum recursion depth exceeded

**RecursionError: maximum recursion depth exceeded.** This is recursion's version of Day 3's infinite loop — except an infinite `while` loop just hangs forever, while Python tracks the call stack's size and refuses to let it grow without limit, so a broken recursion crashes cleanly instead of freezing your program.

### Recursion isn't just for numbers

The "smaller version of the same job" can be a smaller list, a smaller string, a smaller anything.

In [ ]:
def sum_list(numbers):
    if not numbers:                      # base case - an empty list sums to 0
        return 0
    return numbers[0] + sum_list(numbers[1:])   # first item + the sum of the rest

print(sum_list([1, 2, 3, 4, 5]))
print(sum_list([]))

Each call peels off `numbers[0]` and hands the rest, `numbers[1:]`, to itself. The list gets one item shorter every call, so it's guaranteed to eventually hit the base case: an empty list.

In [33]:
# Explain // operation
# In Python, the `//` operator is used for floor division. 
# It divides two numbers and returns the largest integer less than or equal to the result. 
# This means that it effectively "rounds down" to the nearest whole number.
# Example:
a = 1234 // 10
print(a)  # Output: 123, because 1234 divided by 10 is 123.4, and the floor division returns 123.


123


In [34]:
def sum_digits(n):
    if n < 10:              # base case - a single digit sums to itself
        return n
    return n % 10 + sum_digits(n // 10)    # last digit + the sum of the rest

print(sum_digits(1234))    # 1 + 2 + 3 + 4

10


### Recursion vs. a loop — the same answer, two shapes

Anything recursion can do, a loop can also do — usually with less overhead, since every recursive call takes up its own space on the call stack, while a loop just keeps reusing the same space.

In [35]:
def factorial_loop(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

print(factorial_loop(5))

120


Recursion earns its place when a job is *naturally* described in terms of a smaller version of itself — nested data like a business's expense categories is the clearest example, and you'll see one at the end of this section. When a loop and recursion would both work equally well, reach for the loop; it's usually easier to read and cheaper to run.

One more example worth seeing, because it shows recursion's dark side too:

In [36]:
def fibonacci(n):
    if n <= 1:                              # base case - the first two numbers
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)    # each number is the sum of the two before it

for i in range(8):
    print(fibonacci(i), end=" ")

0 1 1 2 3 5 8 13 

This works, but watch what it's actually doing: `fibonacci(5)` calls `fibonacci(4)` and `fibonacci(3)` — and `fibonacci(4)` *also* calls `fibonacci(3)` separately. The same smaller answers get recalculated over and over, and the number of calls explodes as `n` grows. It's correct, and it's slow. Fixing that (without giving up recursion) is a lesson for later in the course — for now, just recognise that recursion being elegant and recursion being efficient are two different questions.

### Real use case: a P&L with nested expense categories

A **P&L** (profit and loss statement) shows what a business earned, what it spent, and what's left over as profit. Expenses are grouped into categories, which have sub-categories, which can have their own — and every business nests them differently.

You can't know in advance how deep the nesting goes, so you can't write the right number of loops. Recursion doesn't care: every category is handled the same way, however deep. Below, a single expense is just a number, and a category is a dictionary of whatever it contains.

In [2]:
expenses = {
    "rent": 80000,
    "salaries": {
        "cashiers": 120000,
        "manager": 90000,
    },
    "marketing": {
        "online": {
            "facebook_ads": 25000,
            "google_ads": 15000,
        },
        "flyers": 5000,
    },
    "utilities": 18000,
}

def total_cost(item):
    if type(item) != dict:            # base case - a single expense, just return it
        return item
    amount = 0
    for child in item.values():       # a category - add up everything inside it
        amount += total_cost(child)
    return amount

revenue = 500000
total_expenses = total_cost(expenses)

print("Revenue:       ", revenue)
print("Total expenses:", total_expenses)
print("Profit:        ", revenue - total_expenses)
print()
print("Marketing alone:", total_cost(expenses["marketing"]))

Revenue:        500000
Total expenses: 353000
Profit:         147000

Marketing alone: 45000


### Breaking it down, step by step

`total_cost` asks one question about whatever it's handed: **is this a single expense (a number), or a category (a dictionary)?**

- A number is the **base case** — just return it.
- A dictionary is the **recursive case** — call `total_cost` on each thing inside it, and add up what comes back.

Here is every call the program makes. Each indent is a call that its parent is waiting on:

```
total_cost(expenses)                   category -> check its 4 items
│
├── "rent": 80000                      number   -> returns 80000
│
├── "salaries"                         category -> check its 2 items
│   ├── "cashiers": 120000             number   -> returns 120000
│   └── "manager": 90000               number   -> returns 90000
│                                      salaries returns 120000 + 90000 = 210000
│
├── "marketing"                        category -> check its 2 items
│   ├── "online"                       category -> check its 2 items
│   │   ├── "facebook_ads": 25000      number   -> returns 25000
│   │   └── "google_ads": 15000        number   -> returns 15000
│   │                                  online returns 25000 + 15000 = 40000
│   └── "flyers": 5000                 number   -> returns 5000
│                                      marketing returns 40000 + 5000 = 45000
│
└── "utilities": 18000                 number   -> returns 18000

total_cost(expenses) returns 80000 + 210000 + 45000 + 18000 = 353000
```

Which gives the P&L:

| Line | Amount (Rs) |
|---|---|
| Revenue | 500,000 |
| Total expenses | 353,000 |
| **Profit** | **147,000** |

Three things to notice:

1. **The function never needed to know how deep the data goes.** `facebook_ads` sits three levels down, and it was reached without writing a single extra loop. Split Facebook ads into separate campaigns tomorrow — a fourth level — and the same function still works, unchanged.
2. **Every call has its own `amount`.** When `total_cost(salaries)` starts its own `amount = 0`, it doesn't wipe out the `amount` that `total_cost(expenses)` is still building up. That's yesterday's local scope at work: each call gets its own private variables.
3. **The totals are added on the way back up.** Just like `sum_to`, nothing gets added until the deepest calls return. `online` has to finish before `marketing` can, and `marketing` has to finish before the grand total can.

The same function works at **any level** — hand it just `expenses["marketing"]` and you get that category's subtotal, with no extra code.

Plenty of business data has this nested shape: an online store's product categories (Electronics → Phones → Android), a company's org chart, or the JSON an API sends back (Day 18). Whenever data contains smaller copies of itself, recursion is the natural tool.

---
# 2. Decorators — adding behaviour to a function without changing it

A decorator wraps extra behaviour — printing a log, timing, checking a login — around a function, without touching the function's own code. To get there, you need two ideas first.

### Idea 1: a function is a value

A function can be stored in a variable and passed around, like a number or a list. You already did this: `key=lambda ...` handed a function to `.sort()`.

In [ ]:
# IDEA 1: A FUNCTION IS A VALUE
# ------------------------------------------------------------
# A variable can hold a number, some text, a list... and also a function.
# The function's name WITHOUT brackets is the function itself.
# Adding brackets () is what RUNS it.

def shout(text):
    return text.upper()

print(shout("hello"))    # shout("hello") -> brackets: RUN it and get "HELLO"
print(shout)             # shout          -> no brackets: the function itself, not run

# Because a function is just a value, you can give it a second name.
# yell and shout are now two labels on the SAME function (like b = a on Day 4).
yell = shout
print(yell("hello"))     # runs the same function -> "HELLO"

# And because it's a value, you can hand it to another function as an argument,
# exactly like handing over a number. This is what key=lambda does with .sort().
def run_it(some_function, text):
    return some_function(text)       # the receiving function decides WHEN to run it

print(run_it(shout, "salaam"))       # pass shout itself - no brackets!

### Idea 2: a function can create and return another function

In [ ]:
# IDEA 2: A FUNCTION CAN CREATE AND RETURN ANOTHER FUNCTION
# ------------------------------------------------------------
# make_greeter is a "function factory": give it a greeting,
# and it builds and hands back a brand-new function that uses that greeting.

def make_greeter(greeting):              # step 1: e.g. greeting = "Salaam"

    def greet(name):                     # step 2: CREATE a function inside - it does NOT run yet
        return f"{greeting}, {name}!"    # <-- CLOSURE: greet uses greeting, which belongs to
                                         #     the OUTER function. This line is what makes
                                         #     greet a closure.

    return greet                         # step 3: hand back the closure itself (no brackets!)
                                         #         make_greeter is now finished

say_salaam = make_greeter("Salaam")      # say_salaam holds a greet function that says "Salaam"
say_hello = make_greeter("Hello")        # a SECOND, separate greet function that says "Hello"

print(say_salaam("Ali"))                 # step 4: NOW greet runs, with name = "Ali"
print(say_hello("Fatima"))

# The surprising part: greeting was a LOCAL variable of make_greeter,
# and make_greeter finished long ago. Yesterday you learned that local variables
# disappear when a function ends - so how does greet still know "Salaam"?
# Because greet was created inside make_greeter, it carries the variables it
# needs with it, like a backpack. A function + its backpack = a CLOSURE.

# Proof: every closure really does carry its backpack, and you can peek inside.
# (__closure__ is Python's name for the backpack - you'll rarely need it, it's just to see it.)
print(say_salaam.__closure__[0].cell_contents)    # "Salaam" - still stored inside say_salaam
print(say_hello.__closure__[0].cell_contents)     # "Hello"  - its own, separate backpack

In [ ]:
# A business example of Idea 2: a discount "factory".
# Each call to make_discount builds a new function that remembers its own percent.

def make_discount(percent):
    def apply(price):
        return price * (1 - percent / 100)    # <-- CLOSURE: apply uses percent from the
                                              #     outer function, so apply is a closure
    return apply                              # hand back the closure

eid_sale = make_discount(20)       # a function that always takes 20% off
clearance = make_discount(50)      # a function that always takes 50% off

print(eid_sale(1000))              # 800.0
print(clearance(1000))             # 500.0

`greet` remembers `greeting` even after `make_greeter` has finished running. A function that remembers values from where it was created is called a **closure** — and it's exactly what makes decorators work.

### Putting it together: a decorator

A decorator is a function that **takes a function, and returns a new version of it** with extra behaviour wrapped around the original.

In [ ]:
# PUTTING IT TOGETHER: A DECORATOR = IDEA 1 + IDEA 2
# ------------------------------------------------------------

def announce(func):                  # IDEA 1: announce RECEIVES a function (func) as a value
    def wrapper():                   # IDEA 2: it CREATES a new function inside
        print("About to run...")     #         extra behaviour BEFORE
        func()                       # <-- CLOSURE: wrapper uses func from the outer function,
                                     #     so wrapper is a closure that remembers which
                                     #     function it wrapped
        print("Finished.")           #         extra behaviour AFTER
    return wrapper                   # IDEA 2: it HANDS BACK the closure (no brackets!)

def make_tea():
    print("Making tea")

# announce(make_tea) -> pass make_tea itself in (Idea 1 - no brackets on make_tea)
#                    -> get back wrapper, which remembers make_tea (Idea 2 + closure)
# make_tea = ...     -> the name make_tea now points to the wrapped version
make_tea = announce(make_tea)

make_tea()     # really runs wrapper: "About to run...", then "Making tea", then "Finished."

`make_tea` itself never changed — `announce` built a new function around it. Writing `make_tea = announce(make_tea)` every time is clumsy, so Python gives you a shortcut: put `@announce` on the line above the `def`.

In [ ]:
# The @ line is only a shortcut. Right after the def, Python does:
#     make_coffee = announce(make_coffee)
# The same Idea 1 + Idea 2 as the cell above - just less typing.

@announce
def make_coffee():
    print("Making coffee")

make_coffee()

### Making a decorator work for any function

`announce` only works on functions with no arguments, because `wrapper()` takes none. To wrap *any* function, the wrapper uses `*args` and `**kwargs` — this is where yesterday's lesson pays off.

In [ ]:
def announce(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished.")
        return result                # don't forget to pass the result back
    return wrapper

@announce
def add(a, b):
    return a + b

print(add(3, 4))

Two new pieces:

- In the `def wrapper(*args, **kwargs)` line, the stars **collect** whatever arguments came in, just like yesterday's `*args` and `**kwargs`.
- In the call `func(*args, **kwargs)`, the stars do the opposite — they **spread** them back out, so the original function receives exactly what the wrapper received.

And `func.__name__` is simply the function's own name, as text.

### Real use case: timing a slow sales report

`import time` loads Python's built-in time tools — imports are covered properly on Day 9. For now, `time.time()` just gives the current time in seconds.

In [ ]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time() - start:.3f} seconds")
        return result
    return wrapper

@timer
def build_sales_report(order_count):
    revenue = 0
    for order_id in range(order_count):
        revenue += 1500              # pretend every order was worth Rs 1,500
    return revenue

print(build_sales_report(1000000))

### Real use case: only managers can issue refunds

In a POS system, a cashier can ring up sales, but refunds need a manager. Instead of pasting the same permission check into every sensitive function — refunds, big discounts, voiding a sale — write it once as a decorator.

In [ ]:
current_staff = {"name": "Hassan", "role": "cashier"}

def manager_only(func):
    def wrapper(*args, **kwargs):
        if current_staff["role"] != "manager":
            print(f"Denied: {current_staff['name']} is not a manager.")
            return None
        return func(*args, **kwargs)
    return wrapper

@manager_only
def issue_refund(order_id, amount):
    print(f"Refunded Rs {amount} for order {order_id}")

issue_refund("A-1042", 2500)                 # blocked - Hassan is a cashier
current_staff["role"] = "manager"
issue_refund("A-1042", 2500)                 # allowed

### Real use case: an audit log of every sale

A business must be able to answer "who sold what, and for how much?" A decorator can record every call to a function automatically — so nobody can forget to log a sale.

In [ ]:
audit_log = []

def log_transaction(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        audit_log.append(f"{func.__name__}{args} -> {result}")
        return result
    return wrapper

@log_transaction
def sell(item, price, quantity):
    return price * quantity

sell("Mouse", 1500, 2)
sell("Keyboard", 3500, 1)

for entry in audit_log:
    print(entry)

> **Where you'll meet decorators for real:** web frameworks use them to connect a web address to a function (FastAPI's `@app.post("/sales")`, Flask's `@app.route("/home")`), and testing and AI-tooling libraries use them to register functions. You'll *use* ready-made decorators far more often than you'll write your own — but now you know what the `@` is doing.

In [3]:
routes = {}                                   # the app's list: (method, address) -> function

def post(path):                               # a decorator FACTORY - takes the address
    def decorator(func):                      # the real decorator - takes the function
        routes[("POST", path)] = func         # register it (the closure remembers path)
        return func                           # hand the function back UNCHANGED
    return decorator

def get(path):
    def decorator(func):
        routes[("GET", path)] = func
        return func
    return decorator

@post("/sales")                               # same as: create_sale = post("/sales")(create_sale)
def create_sale(item, price, quantity):
    return {"item": item, "total": price * quantity}

@get("/products")
def list_products():
    return ["Mouse", "Keyboard", "Monitor"]

for key, func in routes.items():              # what got registered
    print(key, "->", func.__name__)

('POST', '/sales') -> create_sale
('GET', '/products') -> list_products


Nothing has *run* yet — the decorators only filled in the `routes` list. Now pretend requests arrive from customers' browsers. This is the job the web server does for you:

In [ ]:
def handle_request(method, path, **data):
    func = routes.get((method, path))         # look up who handles this address
    if func is None:
        return {"error": "404 Not Found"}     # nobody registered this address
    return func(**data)                       # run the matching function with the request's data

print(handle_request("POST", "/sales", item="Mouse", price=1500, quantity=2))
print(handle_request("GET", "/products"))
print(handle_request("GET", "/refunds"))

print(create_sale("Keyboard", 3500, 1))       # the function itself still works normally

That's the whole trick. Real FastAPI does the same thing with a lot more polish:

- It reads the request's data and matches it to your parameters using the type hints — `item: str`, `price: int` — and rejects a request with a `422` error if something is missing or the wrong type, before your function ever runs.
- It turns whatever your function returns into JSON, the format web browsers and apps understand.
- It answers `404 Not Found` for an address nobody registered, and `405 Method Not Allowed` for a `get` sent to an address that only accepts `post`.
- It builds an interactive page at `/docs` where you can try every address.

The full FastAPI version is saved next to this notebook as `fastapi_routes.py`. To run it on your own computer: `pip install "fastapi[standard]"`, then `fastapi dev fastapi_routes.py`, and open `http://127.0.0.1:8000/docs`.

---
# 3. Generators — producing values one at a time

A normal function builds its whole answer and hands it back all at once with `return`. A **generator** hands values out **one at a time**, only when asked, using `yield` instead of `return`.

In [6]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i          # hand out one value, then pause right here
        i += 1
        # yield i    

for number in count_up_to(5):
    print(number)

1
2
3
4
5


`yield` is like a `return` that doesn't end the function — it hands out a value and **pauses**. Next time a value is asked for, the function resumes exactly where it left off, with all its variables intact.

Calling a generator function doesn't run its body at all. It gives you a generator object, and `next()` pulls one value out at a time:

In [10]:
gen = count_up_to(3)
print(gen)            # a generator object - no code inside has run yet
print(next(gen))      # runs until the first yield
print(next(gen))      # resumes, runs to the next yield
print(next(gen))

<generator object count_up_to at 0x10bb9f340>
1
2
3


In [11]:
# BREAKS ON PURPOSE - the generator has nothing left to give
print(next(gen))

StopIteration: 

**StopIteration** — the generator ran out of values. A `for` loop quietly watches for this signal and stops, which is why looping over a generator never shows the error.

A generator can also only be walked through **once**:

In [12]:
# WRONG ON PURPOSE - the second list is empty, because the generator is used up
numbers = count_up_to(3)
print(list(numbers))
print(list(numbers))

[1, 2, 3]
[]


### Why bother? Memory.

A list stores every value at once. A generator only ever holds one value at a time. Swap the square brackets of a list comprehension (Day 3's sneak peek) for round brackets and you get a **generator expression**:

In [17]:
import sys

squares_list = [n * n for n in range(1000000)]     # builds all million values now
squares_gen = (n * n for n in range(1000000))      # builds each one only when asked

print("List:     ", sys.getsizeof(squares_list), "bytes")
print("Generator:", sys.getsizeof(squares_gen), "bytes")

# print(next(squares_gen))    # 0
# print(next(squares_gen))    # 1



List:      8448728 bytes
Generator: 200 bytes


Millions of bytes against a couple of hundred — and the generator stays that size no matter how big the range gets.

### Real use case: sending orders in batches

An online store might need to hand 5,000 orders to a courier's system — but the courier only accepts 100 at a time. AI work looks the same: sending records to a model in chunks rather than all at once. A generator hands out one batch at a time.

In [18]:
def batches(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]

orders = ["A-1001", "A-1002", "A-1003", "A-1004", "A-1005", "A-1006", "A-1007"]

for batch in batches(orders, 3):
    print("Sending to courier:", batch)

Sending to courier: ['A-1001', 'A-1002', 'A-1003']
Sending to courier: ['A-1004', 'A-1005', 'A-1006']
Sending to courier: ['A-1007']


### Real use case: invoice numbers

Every sale needs its own invoice number, forever. Because a generator only runs when asked, it can safely contain a `while True:` loop — it never runs away on its own.

In [ ]:
def invoice_numbers():
    number = 1
    while True:                         # infinite ON PURPOSE - safe, it only runs when asked
        yield f"INV-{number:04d}"       # :04d pads the number to four digits: 0001, 0002...
        number += 1

invoices = invoice_numbers()
print(next(invoices))
print(next(invoices))
print(next(invoices))

> **Where you'll meet generators for real:** Python reads huge files line by line this way, without loading the whole file into memory. And when an AI chatbot streams its reply to you word by word in Month 2, the code receiving that reply is looping over values arriving one at a time — the same idea.

---
# 4. Putting it all together: a day at a small shop

Everything from the last two days, working together in one small program. A computer shop's POS till rings up sales, numbers every receipt, and logs each sale automatically. At closing time, it produces the day's P&L.

Look for each tool as you read: a **generator** for receipt numbers, a **decorator** that logs sales, `*args` for any number of items, a **default argument** for the discount, a **lambda** for the report, and **recursion** for the nested expenses.

In [ ]:
catalogue = {
    "Mouse":    {"price": 1500,  "cost": 900},       # what we sell it for, what we paid for it
    "Keyboard": {"price": 3500,  "cost": 2200},
    "Monitor":  {"price": 45000, "cost": 36000},
}

sales_log = []

def receipt_numbers():                               # generator
    n = 1
    while True:
        yield f"R-{n:03d}"
        n += 1

receipts = receipt_numbers()

def record_sale(func):                               # decorator
    def wrapper(*args, **kwargs):
        sale = func(*args, **kwargs)
        sales_log.append(sale)
        return sale
    return wrapper

@record_sale
def checkout(*items, discount=0):                    # *args, plus a default argument
    revenue = 0
    cost = 0
    for item in items:
        revenue += catalogue[item]["price"]
        cost += catalogue[item]["cost"]
    revenue = revenue * (1 - discount / 100)         # discount is a percentage
    return {"receipt": next(receipts), "items": items, "revenue": revenue, "cost": cost}

One new detail in `checkout(*items, discount=0)`: any parameter written **after** `*items` can only be given by name — `discount=10` — because `*items` swallows every plain value you pass. That's exactly what you want here: no one can accidentally pass a discount as if it were an item.

Now the shop opens, and three customers come in:

In [ ]:
print(checkout("Mouse", "Keyboard"))
print(checkout("Monitor", discount=10))          # a regular customer gets 10% off
print(checkout("Mouse", "Mouse", "Mouse"))

> **Run the setup cell again before re-running this one.** `sales_log` lives outside the functions, so running the checkout cell twice records the same three sales twice — a small, real example of why yesterday's scope lesson warned about shared global state.

Closing time. First, the day's sales from biggest to smallest, then the P&L:

- **Revenue** — money taken from customers
- **Cost of goods sold** — what the shop paid for the items it sold
- **Gross profit** — revenue minus cost of goods sold
- **Operating expenses** — the costs of running the shop at all: rent, staff, electricity
- **Net profit** — what's actually left: gross profit minus operating expenses

In [ ]:
print("Sales, biggest first:")
for sale in sorted(sales_log, key=lambda s: s["revenue"], reverse=True):     # lambda
    print(" ", sale["receipt"], sale["revenue"])

shop_expenses = {                                    # nested - recursion handles it
    "rent": 3000,
    "staff": {"cashier": 2500, "cleaner": 800},
    "utilities": {"electricity": 900, "internet": 300},
}

def total_cost(item):                                # recursion
    if type(item) != dict:
        return item
    amount = 0
    for child in item.values():
        amount += total_cost(child)
    return amount

revenue = 0
cost_of_goods = 0
for sale in sales_log:
    revenue += sale["revenue"]
    cost_of_goods += sale["cost"]

gross_profit = revenue - cost_of_goods
operating_expenses = total_cost(shop_expenses)
net_profit = gross_profit - operating_expenses

print()
print("--- P&L for today ---")
print("Revenue:           ", revenue)
print("Cost of goods sold:", cost_of_goods)
print("Gross profit:      ", gross_profit)
print("Operating expenses:", operating_expenses)
print("Net profit:        ", net_profit)

Rs 50,000 came through the till, but the shop only kept Rs 700 of it. Most of the money went on buying the stock it sold — and almost all of that came from the Monitor sale, which had a thin margin *and* a discount. That's the kind of question a P&L exists to answer, and it took a handful of small functions to produce one.

---
# 5. Practice set: two interview classics

### A calculator function

In [ ]:
def calculate(a, b, operation):
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        if b == 0:
            return "Cannot divide by zero"
        return a / b
    else:
        return "Unknown operation"

print(calculate(10, 5, "+"))
print(calculate(10, 5, "/"))
print(calculate(10, 0, "/"))
print(calculate(10, 5, "^"))

### A prime checker

A **prime number** is an integer greater than `1` that has exactly two factors: `1` and itself.

The `is_prime(n)` function checks this step by step:

- If `n < 2`, it returns `False` because `0` and `1` are not prime.
- It tests every number from `2` to `n - 1`.
- If `n % i == 0`, then `n` divides evenly by another number, so it is not prime.
- If no divisor is found, it returns `True`.

For example, `7` is prime because it is not evenly divisible by `2`, `3`, `4`, `5`, or `6`. However, `8` is not prime because `8 % 2 == 0`.

The loop checks numbers from `1` through `19` and prints only those for which `is_prime(number)` returns `True`.

In [ ]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

for number in range(1, 20):
    if is_prime(number):
        print(number, end=" ")

The moment a factor turns up, `return False` leaves the function immediately — no need to keep checking. Same "stop as soon as you know the answer" instinct as `break` on Day 3, just via `return` instead.

> This checks every number up to `n - 1`. A faster version only needs to check up to the square root of `n` — not worth the complexity at these sizes, but worth knowing it exists once your numbers get large.

---
# 6. Your turn

**1.** Given `[("Ali", 22), ("Zara", 19), ("Hamza", 25)]`, sort it by age using `.sort()` and a `lambda`, then print it.

**2.** Write a recursive function `count_letters(word)` that returns how many letters a word has, without using `len()`. What is your base case?

**3.** Write `sum_digits(n)` recursively so that `sum_digits(1234)` returns `10`. (Hint: `n % 10` gives the last digit, `n // 10` drops it — you already have this one above, but write it yourself before checking.)

**4.** Write `is_prime(n)`, then use it to print every prime number between 1 and 30.

**5.** Write a decorator `shout` that makes any function which returns text return it in UPPERCASE instead. Test it on a function `greet(name)` that returns `f"hello {name}"`.

**6.** Write a generator `evens(limit)` that yields the even numbers from 2 up to `limit`. Loop over `evens(10)` and print each one.

In [ ]:
# Your practice space

---
### Today you learned

- `lambda parameters: expression` is a one-line, unnamed function — mainly useful as a throwaway `key=` argument for `sorted()`, `.sort()`, `max()` and `min()`
- The `key=` lambda can **calculate** something, like `price * units_sold`, not just pick out a field
- Recursion needs a **base case** and a **recursive case** that moves toward it; without a base case you get `RecursionError`
- Recursion is the natural tool for nested data — folders, product categories, a P&L's expense categories
- A **decorator** takes a function and returns a wrapped version with extra behaviour — logging, timing, permission checks; `@name` above a `def` is the shortcut
- A **generator** uses `yield` to hand out values one at a time, only when asked — it saves memory, handles batches, and can safely run forever
- Any parameter written after `*args` can only be passed by name

**Next:** list and dictionary comprehensions, then `map()`, `filter()` and modules.